# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata_obj = dataset.metadata
print(f"{metadata_obj.name}: {metadata_obj.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List all record sets and their fields by `@id`
for rs in dataset.metadata.record_sets:
    print(f"RecordSet @id: {rs.id}, name: {rs.name}")
    if hasattr(rs, 'fields') and rs.fields:
        print("  Fields:")
        for f in rs.fields:
            print(f"    - {f.id} ({f.name})")
    elif hasattr(rs, 'columns') and rs.columns:
        print("  Columns:")
        for c in rs.columns:
            print(f"    - {c.id} ({c.name})")
    else:
        print("  No fields or columns listed in schema.")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Extract data from each record set
# We'll collect all record sets' `@id`s
record_sets = [rs.id for rs in dataset.metadata.record_sets]
dataframes = {}

for record_set in record_sets:
    try:
        records = list(dataset.records(record_set=record_set))
        if records:
            dataframes[record_set] = pd.DataFrame(records)
            print(f"Loaded {len(dataframes[record_set])} records for RecordSet: {record_set}")
        else:
            print(f"No records found for RecordSet: {record_set}")
    except Exception as e:
        print(f"Could not load records for RecordSet: {record_set} ({e})")

# For this exploration, select the first available record set (if any)
selected_record_set_id = None
if dataframes:
    selected_record_set_id = next(iter(dataframes.keys()))
    print(f"\nSample columns for RecordSet {selected_record_set_id}:")
    print(dataframes[selected_record_set_id].columns.tolist())
    display(dataframes[selected_record_set_id].head())
else:
    print("No available dataframes to display.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
import numpy as np

# For demonstration, pick a numeric field from the selected dataframe (if available)
if selected_record_set_id is not None:
    df = dataframes[selected_record_set_id]
    # Attempt to auto-detect a numeric field
    numeric_candidates = df.select_dtypes(include=[np.number]).columns.tolist()
    if numeric_candidates:
        numeric_field = numeric_candidates[0]
        print(f"Using numeric field: {numeric_field}")

        threshold = df[numeric_field].quantile(0.75)  # e.g., upper quartile
        filtered_df = df[df[numeric_field] > threshold].copy()
        print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
        display(filtered_df.head())

        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Attempt to group by a string/categorical field if present
        group_candidates = df.select_dtypes(include=['object']).columns.tolist()
        group_field = None
        if group_candidates:
            group_field = group_candidates[0]
            grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
            print(f"Grouped data by {group_field}:")
            display(grouped_df.head())
    else:
        print("No numeric fields found in the selected record set.")
else:
    print("No dataframe selected for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualization: Histogram and boxplot of the numeric field
if selected_record_set_id is not None and numeric_candidates:
    plt.figure(figsize=(12,4))
    plt.subplot(1,2,1)
    sns.histplot(data=df, x=numeric_field, kde=True)
    plt.title(f'Histogram of {numeric_field}')
    
    plt.subplot(1,2,2)
    sns.boxplot(x=df[numeric_field])
    plt.title(f'Boxplot of {numeric_field}')
    plt.tight_layout()
    plt.show()

# If also grouped data available, show group means
if selected_record_set_id is not None and 'grouped_df' in locals() and group_field is not None:
    plt.figure(figsize=(8,4))
    grouped_df[numeric_field].plot(kind='bar')
    plt.ylabel(f'Mean {numeric_field}')
    plt.title(f'Mean {numeric_field} by {group_field}')
    plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.